# Final project walkthrough: S&P 500 direction prediction

## 1. Project overview

This notebook documents the final implementation of the project. The objective is to predict whether the S&P 500 will move up on the next trading day, using market-derived features and a reproducible MLOps workflow.

The early work on this topic was exploratory, but the final version is organized as a Kedro project. Kedro now controls the data and modelling flow, MLflow tracks experiments and model versions, SHAP supports model interpretation, and Evidently monitors data drift. FastAPI and Docker provide the serving layer, while pytest is used to validate the implementation.

I keep the notebook deliberately small. It reads the outputs already produced by the project and acts as a walkthrough for the final report; it does not duplicate the production pipeline.

In [ ]:
from pathlib import Path
import json

import pandas as pd
from IPython.display import Image, display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pyproject.toml").exists():
    raise RuntimeError("Run this notebook from the project root.")

## 2. Kedro pipeline registry

The project is divided into modular Kedro pipelines so that each stage has one clear responsibility. The registered pipelines are:

- `data_quality`: checks the structure and basic quality of the raw market data.
- `data_cleaning`: prepares a consistent primary dataset.
- `data_feat_engineering`: creates the technical and return-based predictors.
- `data_split`: produces chronological train, validation and test sets.
- `model_train`: trains the Logistic Regression baseline.
- `model_train_challenger`: trains the Random Forest challenger.
- `model_predict` and `model_predict_challenger`: evaluate the two models on held-out test data.
- `model_explainability`: creates the SHAP artefacts for the registered candidate model.
- `data_expectations`: runs additional validation checks on raw and feature data.
- `data_drift`: compares older and newer feature distributions with Evidently.

In practical terms, data moves from raw validation and cleaning into feature engineering, then into the time-based split. The two model branches train and predict from the same split datasets, which makes their comparison consistent. Explainability, expectations and drift are separate reporting pipelines, so they can be run when their evidence needs to be refreshed.

In [ ]:
registry_path = Path("src/sp500_mlops_pipeline/pipeline_registry.py")
print(f"Pipeline registry: {registry_path}")

## 3. Final feature dataset

At this stage, the raw market data has already been transformed into the final feature dataset used by the Kedro pipeline. I load it here only to inspect the structure and confirm that the notebook remains aligned with the production workflow.

In [ ]:
feature_path = Path("data/04_feature/sp500_feature_data.csv")
feature_data = pd.read_csv(feature_path, parse_dates=["Date"])

print("Shape:", feature_data.shape)
print("Date range:", feature_data["Date"].min().date(), "to", feature_data["Date"].max().date())
print("Columns:", feature_data.columns.tolist())

The current modelling pipeline uses eight features. They capture daily returns, trend, momentum, recent volatility and changes in trading volume.

In [ ]:
MODEL_FEATURES = [
    "simple_return", "log_return", "sma_10", "sma_20",
    "sma_ratio_10", "rsi_14", "volatility_10", "volume_change",
]

feature_data[["Date", *MODEL_FEATURES, "target_next_day_up"]].head()

## 4. Train, validation and test split

The split is chronological: the oldest 70% is used for training, the next 15% for validation, and the newest 15% for final testing. Shuffling would allow information from later market periods to leak into earlier training data, so it is not appropriate here.

The training set is used to fit the models, the validation set supports model comparison and development decisions, and the test set remains the final out-of-sample check. The following cell inspects the datasets written by Kedro rather than recreating the split.

In [ ]:
split_rows = []
for split in ("train", "val", "test"):
    X = pd.read_csv(f"data/05_model_input/X_{split}.csv")
    dates = pd.read_csv(f"data/05_model_input/dates_{split}.csv", parse_dates=["Date"])
    split_rows.append({
        "split": split,
        "rows": len(X),
        "features": X.shape[1],
        "start": dates["Date"].min().date(),
        "end": dates["Date"].max().date(),
    })

pd.DataFrame(split_rows)

## 5. Model training

Logistic Regression is the baseline model in `model_train`. It is a useful first model because it is fast, stable and easy to interpret. The pipeline includes feature scaling before fitting the classifier.

Random Forest is trained separately in `model_train_challenger`. Its role is to test whether a non-linear tree ensemble improves on the simpler baseline. The training logic stays inside the Kedro nodes; repeating it in this notebook would create a second version of the workflow and make the documentation harder to maintain.

## 6. MLflow tracking and model registry

Both training branches record parameters, metrics, feature names and model artefacts in MLflow. This gives each experiment a traceable run and makes it possible to compare results without relying on notebook state.

The model used by the explainability and serving layers is registered as `sp500_direction_model`. The alias `candidate_champion` points those consumers to the selected model version. This separates model selection from deployment code: the API loads the alias instead of hard-coding a version number.

## 7. Explainability with SHAP

Accuracy alone does not show how a market model reaches its predictions. The `model_explainability` pipeline uses SHAP to estimate how each input feature contributes to the Logistic Regression output. The summary plot shows the direction and spread of contributions, while the bar chart gives a simpler global ranking.

In [ ]:
shap_dir = Path("data/08_reporting/shap")
display(Image(filename=str(shap_dir / "logistic_regression_v2_summary_plot.png"), width=850))
display(Image(filename=str(shap_dir / "logistic_regression_v2_feature_importance_bar.png"), width=750))

In [ ]:
with (shap_dir / "logistic_regression_v2_explainability_summary.json").open(encoding="utf-8") as file:
    shap_summary = json.load(file)

pd.DataFrame(shap_summary["top_features"])

## 8. Data drift monitoring

The drift pipeline sorts the feature dataset by date and uses the oldest 70% as reference data and the newest 30% as current data. Evidently then compares the eight production features across the two periods.

This is not a claim that the model has failed. It is an operational signal that the market data seen more recently may differ from the data used as the historical reference, which is useful when deciding whether to investigate performance or retrain the model.

In [ ]:
drift_dir = Path("data/08_reporting/drift")
with (drift_dir / "data_drift_summary.json").open(encoding="utf-8") as file:
    drift_summary = json.load(file)

print("Reference rows:", drift_summary["reference_rows"])
print("Current rows:", drift_summary["current_rows"])
print("Dataset drift detected:", drift_summary["dataset_drift"])
print("Drifted features:", f'{drift_summary["drifted_columns_count"]}/{drift_summary["number_of_features"]}')
print("Full HTML report:", drift_dir / "data_drift_report.html")

In [ ]:
pd.DataFrame(drift_summary["column_drift"]).T[
    ["drift_detected", "drift_score", "method", "threshold"]
]

## 9. API and Docker serving

The selected MLflow model is served through FastAPI. The API loads the `candidate_champion` model when the application starts and exposes three main Swagger endpoints:

- `GET /health` checks that the service and model are available.
- `GET /model-info` returns the model identity, version and expected features.
- `POST /predict` validates the eight input features and returns the predicted class and probability.

The deployment is packaged in the Docker image `sp500-direction-api`. The container runs the FastAPI application with Uvicorn and connects to the MLflow tracking server to resolve the registered model alias. Swagger documentation is available from the standard FastAPI `/docs` route while the service is running.

## 10. Final validation

The final validation run completed with **81 passed tests and 11 warnings**. A full `kedro run` also completed successfully, confirming that the default data preparation, training and prediction workflow executes end to end. I used Kedro-Viz to inspect the pipeline graph and check that the datasets and model branches were connected as intended.

The main outcome of the project is not only a classifier. The final version connects data validation, feature engineering, chronological evaluation, experiment tracking, explainability, drift monitoring and model serving in one Kedro-based workflow. Keeping this notebook focused on produced artefacts makes it useful for the report without turning it into a second implementation.